# Q11.
```{admonition}
:class: note
Repeat the previous exercise, but now fit a nonlinear AR model by "flattening" the short sequences produced for the RNN model.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [3]:
import torch
from torchinfo import summary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
nyse = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/NYSE.csv', parse_dates=['date'], index_col=['date']).sort_index().drop(columns=['day_of_week','train'])

In [5]:
nyse_train, nyse_test = train_test_split(nyse, shuffle=False, train_size=4276)

In [6]:
scaler = StandardScaler()
nyse_train[nyse.columns] = scaler.fit_transform(nyse_train)
nyse_test[nyse.columns] = scaler.transform(nyse_test)
nyse_scaled = pd.concat([nyse_train,nyse_test])
nyse_scaled.head(3)

,DJ_return,log_volume,log_volatility
date,,,
1962-12-03,-0.559615,0.193763,-3.982432
1962-12-04,0.959704,1.552668,-2.240505
1962-12-05,0.468531,2.328698,-2.134713


In [7]:
nyse_lag = pd.concat([nyse_scaled]+[nyse_scaled.shift(lag).add_suffix(f'_lag{lag}') for lag in range (1,6)], axis=1).dropna()
nyse_lag.head(3)

,DJ_return,log_volume,log_volatility,DJ_return_lag1,log_volume_lag1,log_volatility_lag1,DJ_return_lag2,log_volume_lag2,log_volatility_lag2,DJ_return_lag3,log_volume_lag3,log_volatility_lag3,DJ_return_lag4,log_volume_lag4,log_volatility_lag4,DJ_return_lag5,log_volume_lag5,log_volatility_lag5
date,,,,,,,,,,,,,,,,,,
1962-12-10,-1.347250,0.629963,-1.132250,0.062892,0.244084,-2.213740,-0.435955,0.963315,-2.085623,0.468531,2.328698,-2.134713,0.959704,1.552668,-2.240505,-0.559615,0.193763,-3.982432
1962-12-11,0.007932,0.002680,-1.265313,-1.347250,0.629963,-1.132250,0.062892,0.244084,-2.213740,-0.435955,0.963315,-2.085623,0.468531,2.328698,-2.134713,0.959704,1.552668,-2.240505
1962-12-12,0.408248,0.059592,-1.309001,0.007932,0.002680,-1.265313,-1.347250,0.629963,-1.132250,0.062892,0.244084,-2.213740,-0.435955,0.963315,-2.085623,0.468531,2.328698,-2.134713


In [8]:
Y = nyse_lag['log_volume'].to_numpy().astype(np.float32)
X = nyse_lag.drop(columns=nyse_scaled.columns).to_numpy().astype(np.float32)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, shuffle=False, train_size=4276)

In [11]:
torch.manual_seed(1728)

class NYSEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.Sequential(
            nn.Linear(15,32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32,1)
        )

    def forward(self, x):
        return torch.flatten(self.rnn(x))

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_t = torch.tensor(X_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32)
nyse_train = TensorDataset(X_train_t,y_train_t)

train_loader = DataLoader(
    nyse_train,
    batch_size=64,
    shuffle=False
)

X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test,dtype=torch.float32).to(device)

In [13]:
nyse_model = NYSEModel().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(nyse_model.parameters(),lr=0.001)
epochs = 100

for epoch in range(epochs):
    nyse_model.train()
    epoch_loss=0

    for X_batch, y_batch in train_loader:
        mses = nyse_model(X_batch.to(device))
        loss = criterion(mses,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss+=loss.item()
        
    nyse_model.eval()
    with torch.no_grad():
        preds = nyse_model(X_test_t)
        mse = mean_squared_error(y_test,preds.cpu().numpy())
        r2 = r2_score(y_test,preds.cpu().numpy())
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={epoch_loss/len(train_loader):.4f}, mse={mse:.4f}, r2={r2:.4f}')

Epoch 10: loss=0.4955, mse=0.6346, r2=0.4125
Epoch 20: loss=0.4722, mse=0.6249, r2=0.4214
Epoch 30: loss=0.4586, mse=0.6225, r2=0.4236
Epoch 40: loss=0.4626, mse=0.6214, r2=0.4246
Epoch 50: loss=0.4594, mse=0.6203, r2=0.4257
Epoch 60: loss=0.4672, mse=0.6219, r2=0.4242
Epoch 70: loss=0.4578, mse=0.6207, r2=0.4253
Epoch 80: loss=0.4636, mse=0.6215, r2=0.4246
Epoch 90: loss=0.4562, mse=0.6213, r2=0.4247
Epoch 100: loss=0.4536, mse=0.6228, r2=0.4234


In [15]:
nyse_model.eval()

with torch.no_grad():
    pred = nyse_model(X_test_t)
    mse = mean_squared_error(y_test,preds.cpu().numpy())
    r2 = r2_score(y_test,preds.cpu().numpy())
print(f'Test MSE={mse:.4f}, Test R2={r2:.4f}')

Test MSE=0.6228, Test R2=0.4234
